# Phase 2 - Notebook 3.1fix (Manual Labeling 2016)

Este notebook esta hecho para etiquetado manual rapido y estable sobre **USDJPY 2016 H1**.

Objetivo:
- Navegar el ano completo por ventanas.
- Elegir inicio/fin del segmento manualmente.
- Guardar con 1 click y auto-persistencia.
- Evitar perder avance si se cae el kernel.


## Flujo recomendado

1. Ejecuta todas las celdas en orden.
2. Usa la UI interactiva (`launch_ui()`) para navegar y guardar segmentos.
3. Si tu entorno no soporta widgets, usa el modo fallback por funciones (`view`, `save_segment_by_offsets`).
4. El progreso se guarda automaticamente en `output_manual_labels_2016_fix`.


In [1]:
import uuid
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception:
    widgets = None
    HAS_WIDGETS = False

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)


In [2]:
# =========================
# Config
# =========================
DATA_PATH = Path('./data_2016/parquet_h1/USDJPY.parquet')
SYMBOL = 'USDJPY'

OUTPUT_DIR = Path('./phases/phase02_market_alphabet/output_manual_labels_2016_fix')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ACCEPT_PATH_PARQUET = OUTPUT_DIR / f'{SYMBOL}_accepted.parquet'
ACCEPT_PATH_CSV = OUTPUT_DIR / f'{SYMBOL}_accepted.csv'

# Navegacion
WINDOW_BARS_DEFAULT = 240
WINDOW_STRIDE_DEFAULT = 120

# Grafico
PLOT_WIDTH = 17
PLOT_HEIGHT = 6


In [3]:
# =========================
# Carga de datos + estado
# =========================

def _load_data() -> pd.DataFrame:
    if not DATA_PATH.exists():
        raise FileNotFoundError(f'No existe DATA_PATH: {DATA_PATH}')
    df = pd.read_parquet(DATA_PATH).copy()
    need = ['timestamp', 'open', 'high', 'low', 'close']
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f'Faltan columnas {miss}')

    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    for c in ['open', 'high', 'low', 'close']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df = df.dropna(subset=['timestamp', 'close'])
    df = df.sort_values('timestamp').drop_duplicates('timestamp', keep='last').reset_index(drop=True)
    df['idx'] = np.arange(len(df), dtype=int)
    return df


def _load_or_init_labels() -> pd.DataFrame:
    cols = [
        'row_id', 'symbol', 'start_idx', 'end_idx', 'start_ts', 'end_ts', 'n_bars',
        'label', 'quality', 'notes', 'created_at', 'source'
    ]
    if ACCEPT_PATH_PARQUET.exists():
        d = pd.read_parquet(ACCEPT_PATH_PARQUET).copy()
        for c in cols:
            if c not in d.columns:
                d[c] = np.nan
        return d[cols].copy()
    return pd.DataFrame(columns=cols)


DF = _load_data()
ACCEPTED = _load_or_init_labels()

STATE = {
    'current_window': 0,
    'window_bars': int(WINDOW_BARS_DEFAULT),
    'window_stride': int(WINDOW_STRIDE_DEFAULT),
    'last_action': None,
}


def _now_iso() -> str:
    return datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')


def _persist() -> None:
    ACCEPTED.to_parquet(ACCEPT_PATH_PARQUET, index=False)
    ACCEPTED.to_csv(ACCEPT_PATH_CSV, index=False)


def _clip_idx(i: int) -> int:
    return int(min(max(int(i), 0), len(DF) - 1))


def _window_bounds(win_idx: int) -> tuple[int, int]:
    stride = max(1, int(STATE['window_stride']))
    bars = max(20, int(STATE['window_bars']))
    s = _clip_idx(int(win_idx) * stride)
    e = _clip_idx(s + bars - 1)
    return s, e


def _max_window() -> int:
    stride = max(1, int(STATE['window_stride']))
    return max(0, int(np.ceil((len(DF) - 1) / stride)))


def _nearest_idx(ts_like) -> int:
    ts = pd.to_datetime(ts_like, errors='coerce')
    if pd.isna(ts):
        raise ValueError(f'timestamp invalido: {ts_like}')
    arr = DF['timestamp'].to_numpy()
    pos = int(np.searchsorted(arr, np.datetime64(ts), side='left'))
    if pos <= 0:
        return 0
    if pos >= len(arr):
        return len(arr) - 1
    before = arr[pos - 1]
    after = arr[pos]
    return pos if abs(after - np.datetime64(ts)) < abs(np.datetime64(ts) - before) else pos - 1


print(f'SYMBOL={SYMBOL} | rows={len(DF):,} | accepted={len(ACCEPTED):,}')
print(f'range: {DF.timestamp.min()} -> {DF.timestamp.max()}')
print(f'output: {OUTPUT_DIR}')


SYMBOL=USDJPY | rows=6,224 | accepted=34
range: 2016-01-03 17:00:00 -> 2016-12-30 16:00:00
output: ./phases/phase02_market_alphabet/output_manual_labels_2016_fix


In [4]:
# =========================
# Helpers de etiquetado
# =========================

def _segment_exists(start_idx: int, end_idx: int, label: str) -> bool:
    if ACCEPTED.empty:
        return False
    s, e = (start_idx, end_idx) if start_idx <= end_idx else (end_idx, start_idx)
    m = (
        (ACCEPTED['symbol'].astype(str) == str(SYMBOL))
        & (ACCEPTED['start_idx'].astype(int) == int(s))
        & (ACCEPTED['end_idx'].astype(int) == int(e))
        & (ACCEPTED['label'].astype(str) == str(label))
    )
    return bool(m.any())


def _append_segment(start_idx: int, end_idx: int, label: str, quality: int = 3, notes: str = '', source: str = 'manual_3_1fix'):
    global ACCEPTED
    s = _clip_idx(start_idx)
    e = _clip_idx(end_idx)
    if e < s:
        s, e = e, s

    if _segment_exists(s, e, label):
        print('Segmento duplicado para ese label. No se agrega.')
        return None

    row = {
        'row_id': str(uuid.uuid4()),
        'symbol': SYMBOL,
        'start_idx': int(s),
        'end_idx': int(e),
        'start_ts': DF.loc[s, 'timestamp'],
        'end_ts': DF.loc[e, 'timestamp'],
        'n_bars': int(e - s + 1),
        'label': str(label),
        'quality': int(quality),
        'notes': str(notes),
        'created_at': _now_iso(),
        'source': str(source),
    }
    ACCEPTED = pd.concat([ACCEPTED, pd.DataFrame([row])], ignore_index=True)
    _persist()
    STATE['last_action'] = f"save {label} {s}->{e}"
    print(f"OK save: label={label} idx={s}->{e} bars={row['n_bars']}")
    return row


def undo_last():
    global ACCEPTED
    if ACCEPTED.empty:
        print('No hay filas para deshacer.')
        return
    last = ACCEPTED.iloc[-1]
    ACCEPTED = ACCEPTED.iloc[:-1].reset_index(drop=True)
    _persist()
    STATE['last_action'] = 'undo_last'
    print(f"Undo OK: {last['row_id']} {last['label']} {last['start_idx']}->{last['end_idx']}")


def session_status():
    print('--- Session Status ---')
    print(f'SYMBOL: {SYMBOL}')
    print(f'Rows: {len(DF):,}')
    print(f'Accepted: {len(ACCEPTED):,}')
    if len(ACCEPTED):
        print('')
        print('Labels:')
        print(ACCEPTED['label'].value_counts().to_string())
    if STATE.get('last_action'):
        print('')
        print(f"Last action: {STATE['last_action']}")


def list_segments(label: str = None, tail: int = 30):
    if ACCEPTED.empty:
        print('No hay segmentos guardados.')
        return
    d = ACCEPTED.copy()
    if label is not None:
        d = d[d['label'].astype(str) == str(label)].copy()
    cols = ['created_at', 'label', 'quality', 'start_ts', 'end_ts', 'n_bars', 'notes', 'row_id']
    display(d[cols].tail(int(tail)))


def export_train_ready(min_quality: int = 2):
    d = ACCEPTED.copy()
    if d.empty:
        print('No hay segmentos.')
        return
    d = d[d['quality'].astype(int) >= int(min_quality)].copy()
    out_pq = OUTPUT_DIR / f'{SYMBOL}_train_ready_q{min_quality}.parquet'
    out_csv = OUTPUT_DIR / f'{SYMBOL}_train_ready_q{min_quality}.csv'
    d.to_parquet(out_pq, index=False)
    d.to_csv(out_csv, index=False)
    print('Export OK:')
    print('-', out_pq)
    print('-', out_csv)
    print('rows:', len(d))


In [5]:
# =========================
# Visualizacion + fallback sin widgets
# =========================

def _plot_window(win_idx: int, start_idx: int = None, end_idx: int = None, show_table: bool = False):
    ws, we = _window_bounds(win_idx)
    sl = DF.iloc[ws:we+1].copy()

    plt.figure(figsize=(PLOT_WIDTH, PLOT_HEIGHT))
    plt.plot(sl['timestamp'], sl['close'], color='black', lw=1.1)

    # pintar segmentos ya guardados que cruzan la ventana
    if len(ACCEPTED):
        d = ACCEPTED[(ACCEPTED['end_idx'].astype(int) >= ws) & (ACCEPTED['start_idx'].astype(int) <= we)].copy()
        for _, r in d.iterrows():
            s = _clip_idx(int(r['start_idx']))
            e = _clip_idx(int(r['end_idx']))
            plt.axvspan(DF.loc[s, 'timestamp'], DF.loc[e, 'timestamp'], color='gold', alpha=0.22)

    if start_idx is not None and end_idx is not None:
        s = _clip_idx(start_idx)
        e = _clip_idx(end_idx)
        if e < s:
            s, e = e, s
        plt.axvspan(DF.loc[s, 'timestamp'], DF.loc[e, 'timestamp'], color='deepskyblue', alpha=0.28)
        plt.axvline(DF.loc[s, 'timestamp'], color='blue', lw=1.2)
        plt.axvline(DF.loc[e, 'timestamp'], color='blue', lw=1.2)

    ts0 = DF.loc[ws, 'timestamp']
    ts1 = DF.loc[we, 'timestamp']
    plt.title(f'3.1fix Manual Labeling - {SYMBOL} | win={win_idx} | {ts0} -> {ts1} | accepted={len(ACCEPTED)}')
    plt.xlabel('timestamp')
    plt.ylabel('price')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

    if show_table:
        t = sl[['idx', 'timestamp', 'open', 'high', 'low', 'close']].copy()
        display(t.head(5))
        display(t.tail(5))


def view(win_idx: int = None, show_table: bool = False):
    if win_idx is not None:
        STATE['current_window'] = int(max(0, min(int(win_idx), _max_window())))
    _plot_window(STATE['current_window'], show_table=show_table)


def next_view(step: int = 1, show_table: bool = False):
    w = int(STATE['current_window']) + int(step)
    STATE['current_window'] = int(max(0, min(w, _max_window())))
    _plot_window(STATE['current_window'], show_table=show_table)


def prev_view(step: int = 1, show_table: bool = False):
    w = int(STATE['current_window']) - int(step)
    STATE['current_window'] = int(max(0, min(w, _max_window())))
    _plot_window(STATE['current_window'], show_table=show_table)


def jump_to_date(ts_like, show_table: bool = False):
    i = _nearest_idx(ts_like)
    center = max(0, i - STATE['window_bars'] // 2)
    STATE['current_window'] = int(max(0, min(center // max(1, STATE['window_stride']), _max_window())))
    _plot_window(STATE['current_window'], show_table=show_table)


def save_segment_by_offsets(start_off: int, end_off: int, label: str, quality: int = 3, notes: str = ''):
    ws, we = _window_bounds(STATE['current_window'])
    start_off = int(max(0, start_off))
    end_off = int(max(0, end_off))
    s = _clip_idx(ws + start_off)
    e = _clip_idx(ws + end_off)
    _append_segment(s, e, label=label, quality=quality, notes=notes, source='manual_offsets')
    _plot_window(STATE['current_window'], start_idx=s, end_idx=e, show_table=False)


In [6]:
# =========================
# UI interactiva (3.1fix)
# =========================

def launch_ui():
    if not HAS_WIDGETS:
        print('ipywidgets no disponible. Usa modo fallback con: view(), next_view(), save_segment_by_offsets(...)')
        return None

    label_options = [
        'trend_up', 'trend_down', 'triple_top', 'triple_bottom',
        'range', 'reversal', 'continuation', 'impulse_up', 'impulse_down', 'other'
    ]

    w_win = widgets.IntSlider(value=int(STATE['current_window']), min=0, max=_max_window(), step=1, description='win', continuous_update=False)
    w_bars = widgets.IntSlider(value=int(STATE['window_bars']), min=80, max=720, step=20, description='bars', continuous_update=False)
    w_stride = widgets.IntSlider(value=int(STATE['window_stride']), min=20, max=360, step=10, description='stride', continuous_update=False)

    b_prev10 = widgets.Button(description='-10')
    b_prev = widgets.Button(description='Prev')
    b_next = widgets.Button(description='Next')
    b_next10 = widgets.Button(description='+10')
    b_refresh = widgets.Button(description='Refresh', button_style='info')

    w_jump = widgets.Text(value='2016-06-01 00:00', description='jump_ts', placeholder='YYYY-MM-DD HH:MM')
    b_jump = widgets.Button(description='Jump', button_style='warning')

    w_start_off = widgets.IntSlider(value=10, min=0, max=max(1, int(STATE['window_bars']) - 1), step=1, description='start_off', continuous_update=False)
    w_end_off = widgets.IntSlider(value=120, min=0, max=max(1, int(STATE['window_bars']) - 1), step=1, description='end_off', continuous_update=False)
    b_full = widgets.Button(description='Usar ventana completa')

    w_label = widgets.Combobox(options=label_options, value='trend_up', description='label', ensure_option=False)
    w_quality = widgets.Dropdown(options=[1, 2, 3, 4, 5], value=3, description='quality')
    w_notes = widgets.Text(value='', description='notes', placeholder='detalle opcional')

    b_save = widgets.Button(description='Save segment', button_style='success')
    b_save_next = widgets.Button(description='Save + Next', button_style='success')
    b_undo = widgets.Button(description='Undo last')

    out_plot = widgets.Output()
    out_log = widgets.Output()

    def _log(msg: str):
        with out_log:
            print(msg)

    def _sync_window_limits():
        w_win.max = _max_window()
        if w_win.value > w_win.max:
            w_win.value = w_win.max

    def _sync_offset_limits():
        max_off = max(1, int(STATE['window_bars']) - 1)
        w_start_off.max = max_off
        w_end_off.max = max_off
        if w_start_off.value > max_off:
            w_start_off.value = max_off
        if w_end_off.value > max_off:
            w_end_off.value = max_off

    def _current_abs_range():
        ws, we = _window_bounds(int(w_win.value))
        s = _clip_idx(ws + int(w_start_off.value))
        e = _clip_idx(ws + int(w_end_off.value))
        if e < s:
            s, e = e, s
        return ws, we, s, e

    def _refresh(*_):
        STATE['window_bars'] = int(w_bars.value)
        STATE['window_stride'] = int(w_stride.value)
        STATE['current_window'] = int(w_win.value)

        _sync_window_limits()
        _sync_offset_limits()
        ws, we, s, e = _current_abs_range()

        with out_plot:
            clear_output(wait=True)
            _plot_window(int(w_win.value), start_idx=s, end_idx=e, show_table=False)

        _log(f"win={w_win.value} [{DF.loc[ws, 'timestamp']} -> {DF.loc[we, 'timestamp']}] sel_idx={s}->{e} sel_ts={DF.loc[s, 'timestamp']} -> {DF.loc[e, 'timestamp']}")

    def _save(do_next: bool = False):
        try:
            _, _, s, e = _current_abs_range()
            row = _append_segment(
                s, e,
                label=str(w_label.value),
                quality=int(w_quality.value),
                notes=str(w_notes.value),
                source='manual_ui_3_1fix',
            )
            if row is not None:
                _log(f"saved row_id={row['row_id']}")
            if do_next:
                w_win.value = min(w_win.max, int(w_win.value) + 1)
            _refresh()
        except Exception as ex:
            _log(f'save error: {ex}')

    def _on_prev(_):
        w_win.value = max(w_win.min, int(w_win.value) - 1)
        _refresh()

    def _on_next(_):
        w_win.value = min(w_win.max, int(w_win.value) + 1)
        _refresh()

    def _on_prev10(_):
        w_win.value = max(w_win.min, int(w_win.value) - 10)
        _refresh()

    def _on_next10(_):
        w_win.value = min(w_win.max, int(w_win.value) + 10)
        _refresh()

    def _on_jump(_):
        try:
            i = _nearest_idx(w_jump.value)
            center = max(0, i - int(w_bars.value) // 2)
            w_win.value = int(max(0, min(center // max(1, int(w_stride.value)), _max_window())))
            _refresh()
        except Exception as ex:
            _log(f'jump error: {ex}')

    def _on_full(_):
        ws, we = _window_bounds(int(w_win.value))
        w_start_off.value = 0
        w_end_off.value = max(0, we - ws)
        _refresh()

    def _on_undo(_):
        try:
            undo_last()
            _refresh()
        except Exception as ex:
            _log(f'undo error: {ex}')

    b_prev.on_click(_on_prev)
    b_next.on_click(_on_next)
    b_prev10.on_click(_on_prev10)
    b_next10.on_click(_on_next10)
    b_refresh.on_click(lambda _: _refresh())
    b_jump.on_click(_on_jump)
    b_full.on_click(_on_full)
    b_save.on_click(lambda _: _save(do_next=False))
    b_save_next.on_click(lambda _: _save(do_next=True))
    b_undo.on_click(_on_undo)

    w_win.observe(_refresh, names='value')
    w_bars.observe(_refresh, names='value')
    w_stride.observe(_refresh, names='value')
    w_start_off.observe(_refresh, names='value')
    w_end_off.observe(_refresh, names='value')

    ui = widgets.VBox([
        widgets.HTML('<h3>3.1fix Manual Labeling UI (2016)</h3>'),
        widgets.HBox([w_win, w_bars, w_stride]),
        widgets.HBox([b_prev10, b_prev, b_next, b_next10, b_refresh]),
        widgets.HBox([w_jump, b_jump]),
        widgets.HTML('<b>Seleccion del segmento dentro de la ventana</b>'),
        widgets.HBox([w_start_off, w_end_off, b_full]),
        widgets.HBox([w_label, w_quality]),
        widgets.HBox([w_notes]),
        widgets.HBox([b_save, b_save_next, b_undo]),
        widgets.HTML('<b>Grafico</b>'),
        out_plot,
        widgets.HTML('<b>Log</b>'),
        out_log,
    ])

    display(ui)
    _refresh()

    return {
        'ui': ui,
        'controls': {
            'win': w_win,
            'bars': w_bars,
            'stride': w_stride,
            'start_off': w_start_off,
            'end_off': w_end_off,
            'label': w_label,
        },
        'outputs': {'plot': out_plot, 'log': out_log},
    }


In [ ]:
# Inicio
session_status()
UI = launch_ui()

# Si widgets no funciona, usa por ejemplo:
# view(0)
# save_segment_by_offsets(start_off=20, end_off=180, label='trend_up', quality=3, notes='manual')


--- Session Status ---
SYMBOL: USDJPY
Rows: 6,224
Accepted: 34

Labels:
label
range            7
trend_down       6
trend_up         5
impulse_down     5
triple_bottom    4
triple_top       4
impulse_up       2
trend_           1


## Comandos fallback (sin widgets)

```python
view(0)
next_view()
prev_view(step=2)
jump_to_date('2016-10-15 00:00')

# Guardar usando offsets relativos a la ventana actual
save_segment_by_offsets(start_off=15, end_off=170, label='trend_down', quality=3, notes='patron claro')

# Estado y export
session_status()
list_segments(tail=20)
export_train_ready(min_quality=2)
```
